In [ ]:
############  For the regions identified by Champollion, a classical ICP is run for further study  #############
# Assume the database infomation files already processed and saved
# Assume the distance calculation of the region already done and result available

# load the complete dataset from csv file
# generate isomap and UMAP values of the sub-groups
# save results as isomap and UMAP csv files for Moving Average generation

# select different sub-groups (eg. ctl and SCA1)
# combine with the corresponding dataset INFO
# write out the subject names as csv if needed (eg. ctl_sca1_time12)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [3]:
curProject = 'ataxia'
curRoot = 'C'  # 'C' or 'D'

In [5]:
#######################  create a subject_project_timePoint map  #######################
## needed when the subjects don't have the postfix, and it is needed for Shape analysis
## but this will only work if we have one time point only and we know which one it is

atrilFileName = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril\FPO-SCu-ScCal_right_name07-15-26--174_embeddings.csv"
bioscaFileName = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca\FPO-SCu-ScCal_right_name07-15-26--174_embeddings.csv"
cermoiFileName = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi\FPO-SCu-ScCal_right_name07-15-26--174_embeddings.csv"
try:
    data_atril = pd.read_csv(atrilFileName, index_col=0, header=0)
    data_biosca = pd.read_csv(bioscaFileName, index_col=0, header=0)
    data_cermoi = pd.read_csv(cermoiFileName, index_col=0, header=0)    
except FileNotFoundError as e:
    print(f"Error: {e}")

# Create the map DataFrame using the index of the df
subj_proj_time_map_atril = pd.DataFrame(index=data_atril.index)
subj_proj_time_map_atril['projet'] = 'Atril'   # Add and populate the columns
subj_proj_time_map_atril['time_1'] = 'M0'    
subj_proj_time_map_atril['time_2'] = 'M12' 
subj_proj_time_map_atril['time_3'] = np.nan  

subj_proj_time_map_biosca = pd.DataFrame(index=data_biosca.index)
subj_proj_time_map_biosca['projet'] = 'Biosca'   # Add and populate the columns
subj_proj_time_map_biosca['time_1'] = 'E1'    
subj_proj_time_map_biosca['time_2'] = 'E2' 
subj_proj_time_map_biosca['time_3'] = np.nan  

subj_proj_time_map_cermoi = pd.DataFrame(index=data_cermoi.index)
subj_proj_time_map_cermoi['projet'] = 'Cermoi'   # Add and populate the columns
subj_proj_time_map_cermoi['time_1'] = 'V1'    
subj_proj_time_map_cermoi['time_2'] = 'V2' 
subj_proj_time_map_cermoi['time_3'] = 'V3'  

combined_proj_time_map = pd.concat([subj_proj_time_map_atril, subj_proj_time_map_biosca, subj_proj_time_map_cermoi], axis=0)
#print(subj_proj_time_map_atril.head())

outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\subject_project_timeStep_map.csv'
print(outFileName)
#combined_proj_time_map.to_csv(outFileName)

C:\B_projWIP\proj_ataxia\SCA_INFO\subject_project_timeStep_map.csv


In [7]:
###########  Use the subj_timeStep_map created above to rename a given csv file ###########
## needed when the subjects don't have the prefix and postfix, and it is needed for Shape analysis
## but this will only work if we have one time point only and we know which one it is

############################################################################################
### given the map, the subj index, its hemisphere and time_point, returned the new index ###
def get_final_mapped_id(search_id, hemisphere, time_point, target_map):
    # Determine Prefix
    prefix = 'flip-R' if hemisphere == 'R' else 'L'
    # Determine which column to look in for the postfix
    time_col = f"time_{time_point}"
    
    if search_id in target_map.index:
        # Get the postfix (e.g., 'M0' or 'M12') from the map
        postfix = target_map.at[search_id, time_col]        
        # Handle when the postfix is null/NaN
        if pd.isna(postfix):
            return search_id 
        
        # Return the full name: Prefix + ID + Postfix
        return f"{prefix}{search_id}_{postfix}"
    else:
        return search_id  # If subject isn't in the map, return the prefixed ID as a fallback

################################################################################
### read in map file
map_file_name = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\subject_project_timeStep_map.csv'
map_file = pd.read_csv(map_file_name, index_col=0, header=0)
### read in file to be renamed
in_file = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Manhattan\Atril_Biosca_Cermoi_iso_u_sca_2\FPO-SCu-ScCal_right_name07-15-26--174_embeddings_iso_u.csv'
in_data = pd.read_csv(in_file, index_col=0, header=0)

out_file = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\RENAMED_FPO-SCu-ScCal_right_manhattan.csv'
side = 'R'        # Input
time_idx = 1      # Input (for time_1)


new_index_list = []
out_data = in_data.copy()
for raw_id in in_data.index:   
    new_id = get_final_mapped_id(raw_id, side, time_idx, map_file)
    new_index_list.append(new_id)
out_data.index = new_index_list

print(out_file)
#out_data.to_csv(out_file, index_label='subjID')


C:\B_projWIP\proj_ataxia\Champollion\RENAMED_FPO-SCu-ScCal_right_manhattan.csv


In [9]:
################################################    generation of Isomap    ###############################################
from sklearn.manifold import Isomap

def perform_isomap(dist,numDim,numNeig,metric='euclidean'):
    subjNames = dist.index
    dimNames = np.arange(1,numDim+1)
    columns = [f"iso_dim{i}_neig{numNeig}" for i in dimNames]
    #print(columns)    

    dist_centered = dist.copy()
    if metric != 'precomputed':
        dist_centered = dist.values - dist.values.mean(axis=0)
    iso = Isomap(n_neighbors=numNeig,n_components=numDim,metric=metric).fit_transform(dist_centered)
    iso_DF = pd.DataFrame(iso,index=subjNames,columns=columns)    
    return iso_DF
    
def perform_region_isomap(dist,numDim_iso,numNeig_list_iso,curRegion,writeIsomap,metric='euclidean'):
    iso = pd.DataFrame(index=dist.index)
    print('Generating isomaps.')
    for numNeig in numNeig_list_iso:
        iso_cur = perform_isomap(dist,numDim_iso,numNeig,metric=metric)
        # add to existing df
        iso_DF = pd.DataFrame(iso_cur, index=iso.index)
        iso = pd.concat([iso, iso_DF], axis=1)
    if writeIsomap:  # SAVE Isomaps as csv, for debug
        outName = 'iso_'+curRegion+'_dim_'+str(numDim_iso)+'_neig_'+str(numNeig_list_iso)+'.csv'
        outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u\{outName}'
        print(outFileName)
        iso.to_csv(outFileName,index_label='subjID')
    return iso

In [11]:
###############################   generation of UMAP   ################################
import umap
import random
# to ensure that the results are always the same
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

def perform_UMAP(df,n_comp,n_neighbors,min_dist):
    reducer = umap.UMAP(n_components=n_comp,n_neighbors=n_neighbors,min_dist=min_dist,random_state=SEED)
    embedding = reducer.fit_transform(df)

    columns = []    
    if n_comp == 1:    # Column naming logic
        columns = [f'u_dim{n_comp}_neig{n_neighbors}']
    elif n_comp == 2:
        columns = [f'u_dim{n_comp}_1_neig{n_neighbors}', f'u_dim{n_comp}_2_neig{n_neighbors}']
    else:
        columns = [f'u_dim{n_comp}_{i}_neig{n_neighbors}' for i in range(n_comp)]
    
    embedding_df = pd.DataFrame(embedding, columns=columns)   # Create a DataFrame for the embedding
    embedding_df.index = df.index
    return embedding_df

def perform_region_UMAP(dist,curRegion,numDim_u,numNeig_list_u,writeUMAP):
    umap_results = pd.DataFrame(index=dist.index)
    min_dist = 0.2                # Change this to the desired minimum distance
    print('Generating UMAPs.')
    for n_comp in numDim_u:
        for n_neighbors in numNeig_list_u:  # perform UMAP
            embedding_df = perform_UMAP(dist, n_comp, n_neighbors, min_dist)   # Call the helper function                     
            umap_results = pd.concat([umap_results, embedding_df], axis=1)    # Concatenate to our results dataframe
    if writeUMAP:  # SAVE Umap as csv, for debug
        outName = 'u_'+curRegion+'_dim_'+str(numDim_u)+'_neig_'+str(numNeig_list_u)+'.csv'
        outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u\{outName}'
        print(outFileName)
        umap_results.to_csv(outFileName,index_label='subjID')
    return umap_results

In [57]:
#######################   create iso_u directory to write output if needed   ########################
#outFileDir = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\FPOCalCu_iso_u_with_DB_info"
outFileDir = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\Calc_iso_u_with_DB_info"
print(outFileDir)
os.makedirs(outFileDir, exist_ok=True)


C:\B_projWIP\proj_ataxia\Champollion\AtrilBioscaCermoi_Champollion_Regions\Calc_iso_u_with_DB_info


In [13]:
#######################   read in all_DB_info csv   ########################
infoBaselineName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_BIOSCA_CERMOI_time1_only.csv'
Atril_Biosca_Cermoi_INFO = pd.read_csv(infoBaselineName,index_col=0,header=0)

#print("combined_iso_u columns:", combined_iso_u.columns.tolist())
print("INFO columns:", Atril_Biosca_Cermoi_INFO.columns.tolist())
print(Atril_Biosca_Cermoi_INFO)

### verification of column names ###
#one_iso_u_name = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u\FColl-SRh_left_name06-43-43--210_embeddings_iso_u.csv'
#one_combined_iso_u = pd.read_csv(one_iso_u_name,index_col=0,header=0)
#print("region columns:", one_combined_iso_u.columns.tolist())

INFO columns: ['RANDOMIZATION', 'SCA', 'CAG', 'Sex', 'Age', 'SARA', 'INAS', 'CodeICM', 'Age_onset', 'CCFS', 'Handedness', 'Disease_duration', 'allele1', 'allele2', 'minAllele', 'maxAllele', 'sumAllele']
          RANDOMIZATION  SCA   CAG  Sex        Age  SARA  INAS CodeICM  \
subjID                                                                   
0010001OP             A    2  35.0    1  61.089665   8.0   2.0   ATRIL   
0010002MV             B    2  43.0    2  34.858316  11.5   5.0   ATRIL   
0010003CJ             B    2  39.0    2  36.522930   7.0   2.0   ATRIL   
0010004HV             A    2  43.0    2  32.887064  11.0   4.0   ATRIL   
0010005BC             B    2  35.0    2  66.861054  21.0   5.0   ATRIL   
...                 ...  ...   ...  ...        ...   ...   ...     ...   
00036DC             NaN    0   NaN    1  60.000000   0.0   2.0  CERMOI   
00037CI             NaN    2  38.0    1  48.000000   3.5   4.0  CERMOI   
00038RS             NaN    7  38.0    1  26.000000   0.0 

In [61]:
###############  For record, the debug for merging two df on index  ##############
left_set = set(filtered_combined_iso_u.index)
right_set = set(Atril_Biosca_Cermoi_INFO.index)
common = left_set & right_set
left_only = left_set - right_set
right_only = right_set - left_set
print(common)
print(left_only)
print(right_only)

left_idx = filtered_combined_iso_u.index.astype(str)
right_idx = Atril_Biosca_Cermoi_INFO.index.astype(str)
common = left_idx.intersection(right_idx)
print("Left size:", len(left_idx))
print("Right size:", len(right_idx))
print("Common:", len(common))


NameError: name 'filtered_combined_iso_u' is not defined

In [104]:
#####################################################################################################################
## All SCAs together                                                                                               ##
## for a given region, reading distance files of Atril, Biosca, Cermoi, combine to one distance file               ##
##                  perform isomap and umap for ALL sca's together                                                 ##
##                  combine this to write the shape output and the version with INFO                               ##
## NOTE: the difference of this code from the original Champollion processing is that:                             ##
##       there are min and max, one region at a time, left and right needs to be separated                         ##
##       the index of the distance matrix needs to be processed remove pre and post_fix for the INFO merge to work ##
#####################################################################################################################

numDim_iso = 6                 # MODIFY if needed
numNeig_list_iso = {10,30}     # MODIFY if needed
numDim_u = {1,2}
numNeig_list_u = {10,30}       # MODIFY if needed
curRegion = 'FPOCalCu' # original name: "FPO-SCu-ScCal_right_name07-15-26--174_embeddings"
curType = 'min'
curHem = 'right'

###########  input distance file  ###########
in_dist_name = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\{curRegion}_linux\Isomap\{curType}Dist{curRegion}.txt"
dist_Atril_Biosca_Cermoi = pd.read_csv(in_dist_name,index_col=0,header=0)

###########  generating isomap/umap  ###########
iso = perform_region_isomap(dist_Atril_Biosca_Cermoi,numDim_iso,numNeig_list_iso,curRegion,writeIsomap=False)  # write true to debug iso file
u = perform_region_UMAP(dist_Atril_Biosca_Cermoi,curRegion,numDim_u,numNeig_list_u,writeUMAP=False)            # write true to debug umap file
cur_combined_iso_u = pd.concat([iso,u], axis=1)

###########  filtering by hemisphere  ###########
filtered_combined_iso_u = cur_combined_iso_u
if curHem == "left":
    filtered_combined_iso_u = cur_combined_iso_u[cur_combined_iso_u.index.str.startswith("L")]
else:  # curHemisphere == "right"
    filtered_combined_iso_u = cur_combined_iso_u[cur_combined_iso_u.index.str.startswith("flip-R")]
###########  write out the iso_u file  ###########
combined_iso_u_fileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\{curRegion}_iso_u_{curType}_{curHem}.csv'
filtered_combined_iso_u.to_csv(combined_iso_u_fileName,index_label='subjID')    

###########  merging with DB INFO  ###########
## reformat the subjID index so that the merge with DB INFO below would work
#filtered_combined_iso_u.index = (filtered_combined_iso_u.index
#                            .str.split('_').str[0]                       # remove _anything from the end of the subjID index
#                            .str.replace(r'^(L|flip-R)', '', regex=True))# remove L or flip-R from the front of the subjID index 
filtered_combined_iso_u.index = (filtered_combined_iso_u.index
        .str.split('_').str[0]                    # remove suffix
        .str.replace(r'^flip-R', '', regex=True)  # remove leading flip-R
        .str.replace('flip-', '', regex=False)    # fix middle flip-R → keep R
        .str.replace(r'^L', '', regex=True)       # remove leading L only                                 
)
combined_iso_u_INFO = pd.merge( # the actual merge
filtered_combined_iso_u,
Atril_Biosca_Cermoi_INFO,
left_index=True,  #  subjID is the index name of left df
right_index=True  # subjID is the index name of the right df
)
#print(combined_iso_u_INFO.isna().sum()) # check number of each column if needed

###########  writing out the iso_u file with INFO  ###########  
combined_iso_u_INFO_fileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\{curRegion}_iso_u_INFO_{curType}_{curHem}.csv'
print(combined_iso_u_INFO_fileName)
combined_iso_u_INFO.to_csv(combined_iso_u_INFO_fileName,index_label='subjID')

Generating isomaps.
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\AtrilBioscaCermoi_Champollion_Regions\FPOCalCu_iso_u_INFO_min_right.csv


In [15]:
###############  Define subject lists according to SCA type for distance selection  ################
## NOTE: SCA selection doesn't include ATRIL so that there are equal number of control and patients
##       this could be modified if needed

# 1. Get the relevant subjects
sca_1_ctl = (Atril_Biosca_Cermoi_INFO['SCA'] == 1) | \
            ((Atril_Biosca_Cermoi_INFO['SCA'] == 0) & (Atril_Biosca_Cermoi_INFO['CodeICM'] == 'BIOSCA'))
sca_2_ctl = ((Atril_Biosca_Cermoi_INFO['SCA'] == 2) & (Atril_Biosca_Cermoi_INFO['CodeICM'] != 'ATRIL')) | \
            (Atril_Biosca_Cermoi_INFO['SCA'] == 0) 
sca_2_ctl_with_Atril = (Atril_Biosca_Cermoi_INFO['SCA'] == 2) | (Atril_Biosca_Cermoi_INFO['SCA'] == 0) 
sca_3_ctl = (Atril_Biosca_Cermoi_INFO['SCA'] == 3) | \
            ((Atril_Biosca_Cermoi_INFO['SCA'] == 0) & (Atril_Biosca_Cermoi_INFO['CodeICM'] == 'BIOSCA'))
sca_7_ctl = (Atril_Biosca_Cermoi_INFO['SCA'] == 7) | (Atril_Biosca_Cermoi_INFO['SCA'] == 0) 

# 2. Use the condition to get the index (subject names)
subjects_sca_1_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_1_ctl].index.tolist()
#print(subjects_sca_1_ctl)
subjects_sca_2_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_2_ctl].index.tolist()
print(subjects_sca_2_ctl)
subjects_sca_2_ctl_with_Atril = Atril_Biosca_Cermoi_INFO.loc[sca_2_ctl_with_Atril].index.tolist()
print(subjects_sca_2_ctl_with_Atril
     )
subjects_sca_3_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_3_ctl].index.tolist()
#print(subjects_sca_3_ctl)
subjects_sca_7_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_7_ctl].index.tolist()
#print(subjects_sca_7_ctl)


['001012FJ', '001015VJ', '001017VP', '001019DA', '001020HG', '001021CJ', '001022LM', '001025LJ', '001027RY', '001032SG', '001037GA', '001040BF', '001045PB', '001046CJ', '001049BD', '001054MP', '001055JC', '001057MB', '001058FG', '001059MV', '001060MJ', '001065BC', '001073PM', '001075HJ', '001078PM', '001079LP', '001085BN', '001086CP', '001091MR', '001099GL', '001100PY', '001101JO', '00001PJ', '00002PV', '00004PA', '00007OP', '00008CJ', '00009LN', '00011EG', '00012BM', '00017ML', '00019RP', '00020CT', '00021JA', '00022DA', '00023EA', '00025AY', '00026AD', '00027EF', '00029DP', '00030CA', '00031CP', '00032DL', '00035NR', '00036DC', '00037CI', '00039OV']
['0010001OP', '0010002MV', '0010003CJ', '0010004HV', '0010005BC', '0010006OG', '0010007MA', '0010008CT', '0010009BJ', '0010010DM', '0010011CP', '0010012MC', '0010013AN', '0010014MM', '0010015BV', '0010016VP', '0010017MD', '0010018RE', '0010019MM', '0010020PA', '0010021MA', '0010022MB', '0010023MM', '0010024VN', '0010025RA', '0010026DA', '

In [25]:
################### Debug: test Selection of distance according to SCA ###################
curRegion = 'Calc' #FPOCalCu
sca_targets = [1, 2, 3, 7]

def clean_index(in_list):
    return (in_list
        .str.split('_').str[0]                    # remove suffix
        .str.replace(r'^flip-R', '', regex=True)  # remove leading flip-R
        .str.replace('flip-', '', regex=False)    # fix middle flip-R → keep R
        .str.replace(r'^L', '', regex=True)       # remove leading L only                                 
    )
curType = 'min'
##  input distance file  
in_dist_name = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\{curRegion}_linux\Isomap\{curType}Dist{curRegion}.txt"
dist_Atril_Biosca_Cermoi = pd.read_csv(in_dist_name,index_col=0,header=0)
##  Fix the Index
dist_Atril_Biosca_Cermoi.index = clean_index(dist_Atril_Biosca_Cermoi.index)
##  Fix the Columns
dist_Atril_Biosca_Cermoi.columns = clean_index(dist_Atril_Biosca_Cermoi.columns)

#########################  Selection of subjects  ########################
for sca in sca_targets:

    subjects_select = []
    if sca == 1:
        subjects_select = subjects_sca_1_ctl
    if sca == 2:
        subjects_select = subjects_sca_2_ctl
    if sca == 3:
        subjects_select = subjects_sca_3_ctl
    if sca == 7:
        subjects_select = subjects_sca_7_ctl  
    if sca == 102:
        subjects_select = subjects_sca_2_ctl_with_Atril              
    valid_subjects = dist_Atril_Biosca_Cermoi.index.intersection(subjects_select)
    dist_select = dist_Atril_Biosca_Cermoi.loc[valid_subjects,]
    print(subjects_select)
    print(dist_Atril_Biosca_Cermoi)
    

['001012FJ', '001013LF', '001015VJ', '001019DA', '001020HG', '001022LM', '001029DJ', '001032SG', '001033BM', '001035GA', '001037GA', '001039CD', '001040BF', '001049BD', '001060MJ', '001065BC', '001068RA', '001069DB', '001073PM', '001075HJ', '001076SC', '001077MR', '001079LP', '001083RB', '001084CJ', '001085BN', '001087RN', '001090RD', '001091MR', '001093TL', '001094GE', '001095MP', '001096BC', '001099GL', '001100PY', '001101JO']
            001020HG  0060001PJ    00033DL    00031CP   001070CF  0010008CT  \
subjName                                                                      
001020HG    0.000000   4.274317   5.542100   4.005120  13.301366   5.031721   
0060001PJ   4.274317   0.000000  12.483397   4.216100   7.028737   6.278180   
00033DL     5.542100  12.483397   0.000000  10.071808  24.229611   4.918588   
00031CP     4.005120   4.216100  10.071808   0.000000   9.953379   7.891101   
001070CF   13.301366   7.028737  24.229611   9.953379   0.000000  16.160708   
...           

In [35]:
####################################################################################################
## Same as the isomap/Uamp generation above, BUT for specific SCAs                                ##
## for one region, reading distance files of Atril, Biosca, Cermoi, combine to one distance file  ##
##                 selection on distance according to SCA type                                    ##
##                 perform isomap and umap                                                        ##
##                 combine this iso_u with DB INFO, write out                                     ##
####################################################################################################

###############################################################################################
######################### To fix possible errors in names due to bugs #########################
## Note that this is needed separate from the clean_index funtion below since we need a correct
## index for the isomap/umap final output for further shape MA generation
## This was needed for FPOCalCu because of a bug that add flip- when R in the middle of subjName
##  bug fixed for Calc
def sanitize_original_names(in_list):
    return (in_list
        .str.replace(r'^Flip-', 'flip-', regex=True)        # 1. Fix 'Flip-' -> 'flip-'
        .str.replace(r'(.+)flip-', r'\1', regex=True)      # 2. Remove 'flip-' if in the middle
        .str.strip()                                       # 3. Remove accidental spaces
    )
################################################################################################
##############  To match INFO format, fix subjName format in distance matrixes  ################
## NOTE that there is a bug that add 'flip-' in front of R in the middle of the name as well
## To fix this remove here 'flip-R' from the beginning of the name ONLY, then remove all 'flip-'
## if they are in the middle of the names
## Also remove postfix only 'INFO_merge', when we need to merge isomap with INFO, but for
## shape brnach to generate MA we need the suffix, so it should NOT be removed.
    
def clean_index(in_list):
    return (in_list
        .str.replace(r'_[^_]*$', '', regex=True)  # remove the last _
        .str.replace(r'^flip-R', '', regex=True)  # remove leading flip-R
        .str.replace('flip-', '', regex=False)    # fix middle flip-R → keep R
        .str.replace(r'^L', '', regex=True)       # remove leading L
    )
    
sca_targets = [1, 2, 3, 7]
#sca_targets = [102] # SPECIAL CODE 102: sca 2 Biosca + Control Biosca + Atril

numDim_iso = 6
numNeig_list_iso = {5,30}     # MODIFY if needed
numDim_u = {1,2}
numNeig_list_u = {10,30}       # MODIFY if needed
metric = 'euclidean' # precomputed if use original ICP dist!'cosine'/'manhattan'/'euclidean' ONLY if recalculating embedding
curRegion = 'Calc' # FPOCalCu, original name: "FPO-SCu-ScCal_right_name07-15-26--174_embeddings"
curType = 'min'  # max or min
curHem = 'left' # left or right

##  input distance file  ##
in_dist_name = (rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\{curRegion}_linux\Isomap\{curType}'
               rf'Dist{curRegion}.txt')
dist_Atril_Biosca_Cermoi = pd.read_csv(in_dist_name,index_col=0,header=0)
sanitized_originals = sanitize_original_names(dist_Atril_Biosca_Cermoi.index.astype(str))
# Create the Map using the sanitized names
# Key: The fully stripped ID (Subj_01)
# Value: The corrected original (flip-R_Subj_01_v1)
# We need a map between the sanitized index and the cleaned index:
# The sanitized for writing out isomap/umap files for further shape analysis and MA generation
# The cleaned version for distance SCA filtering and merging with the subject INFO
name_map = dict(zip(clean_index(sanitized_originals), sanitized_originals))


###########  filtering by hemisphere  ###########
if curHem == "left":
    dist_Atril_Biosca_Cermoi = dist_Atril_Biosca_Cermoi.loc[
        dist_Atril_Biosca_Cermoi.index.str.startswith("L"),
        dist_Atril_Biosca_Cermoi.columns.str.startswith("L")
    ]
else:  # curHem == "right"
    dist_Atril_Biosca_Cermoi = dist_Atril_Biosca_Cermoi.loc[
        dist_Atril_Biosca_Cermoi.index.str.startswith("flip-R"),
        dist_Atril_Biosca_Cermoi.columns.str.startswith("flip-R")
    ]
    
# Clean the Index format 
dist_Atril_Biosca_Cermoi.index = clean_index(dist_Atril_Biosca_Cermoi.index)
# Clean the Columns format 
dist_Atril_Biosca_Cermoi.columns = clean_index(dist_Atril_Biosca_Cermoi.columns)

for sca in sca_targets:
    #########################  Selection of subjects  ########################
    subjects_select = []
    if sca == 1:
        subjects_select = subjects_sca_1_ctl
    if sca == 2:
        subjects_select = subjects_sca_2_ctl
    if sca == 3:
        subjects_select = subjects_sca_3_ctl
    if sca == 7:
        subjects_select = subjects_sca_7_ctl  
    if sca == 102:
        subjects_select = subjects_sca_2_ctl_with_Atril              
    valid_subjects = dist_Atril_Biosca_Cermoi.index.intersection(subjects_select)
    ##dist_select = dist_Atril_Biosca_Cermoi.loc[valid_subjects,]
    # The "Square" Selection
    dist_select = dist_Atril_Biosca_Cermoi.loc[valid_subjects, valid_subjects]   

    ###########  generating isomap/umap  ###########
    iso = perform_region_isomap(dist_select,numDim_iso,numNeig_list_iso,curRegion,writeIsomap=False,metric=metric)  # write true to debug iso file
    u = perform_region_UMAP(dist_select,curRegion,numDim_u,numNeig_list_u,writeUMAP=False)            # write true to debug umap file
    cur_combined_iso_u = pd.concat([iso,u], axis=1)

    
    ################################  Modify the index to switch back to the original for MA generation  ##############################
    write_combined = cur_combined_iso_u.copy()
    # This looks up each clean index in our dictionary and replaces it with the original
    write_combined.index = write_combined.index.map(name_map)
    ###########  write out the iso_u file  ###########
    combined_iso_u_fileName = (rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\{curRegion}'
                              rf'_iso_u_{curType}_{curHem}_{metric}_SCA_{sca}.csv')
    write_combined.to_csv(combined_iso_u_fileName,index_label='subjID')  
    ###############################################################################################################################
    
    ###########  merging with DB INFO  ###########
    # Clean the Index format, remove the postfix for the merging process
    cur_combined_iso_u.index = clean_index(cur_combined_iso_u.index)
    combined_iso_u_INFO = pd.merge(
    cur_combined_iso_u,
    Atril_Biosca_Cermoi_INFO,
    left_index=True,  #  subjID is the index name of left df
    right_index=True  # subjID is the index name of the right df
    )
    ###########  writing out the iso_u file with INFO  ###########
    combined_iso_u_INFO_fileName = (rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\AtrilBioscaCermoi_Champollion_Regions\{curRegion}'
                                   rf'_iso_u_INFO_{curType}_{curHem}_{metric}_SCA_{sca}.csv')
    print(combined_iso_u_INFO_fileName)
    combined_iso_u_INFO.to_csv(combined_iso_u_INFO_fileName,index_label='subjID')
    

Generating isomaps.
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for paralle

C:\B_projWIP\proj_ataxia\Champollion\AtrilBioscaCermoi_Champollion_Regions\Calc_iso_u_INFO_min_left_euclidean_SCA_1.csv
Generating isomaps.
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\AtrilBioscaCermoi_Champollion_Regions\Calc_iso_u_INFO_min_left_euclidean_SCA_2.csv
Generating isomaps.
Generating UMAPs.
C:\B_projWIP\proj_ataxia\Champollion\AtrilBioscaCermoi_Champollion_Regions\Calc_iso_u_INFO_min_left_euclidean_SCA_3.csv
Generating isomaps.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Generating UMAPs.
C:\B_projWIP\proj_ataxia\Champollion\AtrilBioscaCermoi_Champollion_Regions\Calc_iso_u_INFO_min_left_euclidean_SCA_7.csv


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
